# archon-emu
## Emulator / Debugger library for Archon Timing scripts
---
Dan Weatherill

University of Oxford

** Jan 17 2017 SAWG Meeting **

# Basic Details

* Written in pure Python with only pure python external dependencies (therefore it is **SLOW** but easy to get running)
* Python 3 **ONLY** (not deliberately broken on 2.7, but no testing and likely invalid constructs)
* Currently almost no nice UI, but can be used for various tasks. I have been using it for
    * development of timing scripts
    * easily viewing timing script from .acf file
    * converting legacy DSP timing scripts to Archon
* Still issues to work out and nice interfaces to make. Please send suggestions/ requests/ (ideally) pull requests  to **daniel.weatherill@physics.ox.ac.uk** and on github issues :-)

 __Available at: <https://github.com/LSSTDESC/archon_emu> __

# Features
1. Load and parse an .acf file into "nice" Python datastructures (`ArchonConfig` class)
2. Parse timing script and assemble a list of time parallel instructions that the timing cores will execute (`ArchonCompiler` class)
3. Step through these instructions and produce a list of states that will be output by timing cores (`ArchonVM` class), optionally with instrumentation showing what subroutine you are in and where it was called from (`ArchonVM_Instrumented` class)
4. Given the list of timing states, extract clock level information (`ArchonSM` class)


# Points to Note
* Some things about the specification of Archon timing are actually not well specified by Archon manual (i.e. these constructs are never mentioned but have been seen in the wild so they must work!). I had to implement most of these so the program would work with the files sent by Paul O'Connor. Quiz:
    * What does this line do?
    
    ```
    StateA; IF MyParameter StateB
    ```
    
    * Or this one?
    
    ``` 
    IF MyParameter MyParameter-- 
    ```
    
    * Is this even a valid construct (It works in archon-emu)?
    ```
    StateA; CALL MySub(MyParameter); MyParameter--
    ```
    
    * This IS a valid construct but not mentioned in Archon manual
    ```
    MyParameter=623 #this is an end of line comment!
    ```
    

# Basic Usage
## installation
use pip or setuptools, i.e.

```
pip install -r requirements.txt
pip install -e .
```

## imports and setup

In [1]:
import warnings 
import importlib
import logging
from archon_emu.acffile import ArchonConfig, StateConfigWarning #.acf parsing
from archon_emu.archon_timing_dsl import ArchonCompiler #compile timing scripts
from archon_emu.archonVM import ArchonVM, ArchonVM_Instrumented #step through timing
#ArchonSM 

#set to logging.DEBUG to see a lot of stuff about compilation
importlib.reload(logging)
logging.basicConfig(level=logging.WARN)    

#you want to disable this warning or it will produce many 
#warnings for a lot of scripts
warnings.simplefilter('ignore',category=StateConfigWarning) 


## load and compile timing information

In [2]:
fl2 = 'seqs/CCD250_e2v_Image.acf'
#load and parse acf file
archon_conf = ArchonConfig.from_filename(fl2)
#compile timing file - can be customized in many ways if desired
compiler = ArchonCompiler.from_archonconfig(archon_conf)
#create a VM to step through timing states
vm  = ArchonVM.from_compiler(compiler)

module, constant, parameter and timing state information is available through property calls on the `ArchonConfig` object, for example:

In [3]:
print(archon_conf.parameters)
#print(archon_conf.constants)
#print(archon_conf.states)
#print(archon_conf.modules)

{'WaiTime': 100, 'Pdepth': 1, 'Tsr9': 51, 'Tsr4': 10, 'Tsrf2': 20, 'Tsr6': 7, 'Tpu': 200, 'PixPre': 10, 'Nclear': 2002, 'Tsrf9': 102, 'Tsrf4': 20, 'Nexpo': 100000, 'RowOver': 0, 'RowSkip': 0, 'Tsr5': 4, 'Tsrf3': 20, 'Tsrf7': 84, 'Tsr7': 42, 'Tsrf6': 14, 'Tsr8': 7, 'ReadDel': 0, 'PixOver': 0, 'LCLK': 0, 'Tsrf5': 8, 'RowRead1': 2048, 'ExpTime': 0, 'Tsr1': 0, 'RowRead2': 0, 'PixSkip': 0, 'Tpa': 250, 'PixRead': 576, 'Tnpcf': 69, 'Tsrf8': 14, 'Twait': 0, 'Tsf': 60, 'Tpc': 500, 'Tsr3': 10, 'Expo': 1, 'Tnpc': 29, 'Tpb': 400, 'Light': 1, 'Npump': 0, 'Tsrf1': 1, 'Tsr2': 10}


In [9]:
#properly organised timing script is also available, which can be useful if only provided with an .acf file
print(archon_conf.timing_script)

Start:
LowCLK; IF !LCLK GOTO Start
IDLE; CALL RCLR(4)
KEEPER; CALL EXPOSE(Expo)
KEEPER; CALL READOUT(Expo)
KEEPER; IF !Nexpo CALL ExpoDONE
KEEPER; GOTO Start
ExpoDONE:
KEEPER; Expo--
KEEPER; RETURN ExpoDONE
RCLR:
IDLE; CALL FCLR(Nclear)
KEEPER; CALL SCLR(PixSkip)
KEEPER; CALL SCLR(PixRead)
KEEPER; CALL SCLR(PixOver)
KEEPER; RETURN RCLR
FCLR:
PFA; PFA(Tpa)
PFB; PFB(Tpb)
PFC; PFC(Tpb)
PFD; PFD(Tpb)
PFE; PFE(Tpb)
PFF; PFF(Tpb)
PFG; PFG(Tpb)
PFH; PFH(Tpb)
PFI; PFI(Tpa)
KEEPER; RETURN FCLR
SCLR:
SF1;SF1(Tsf)
SF2;SF2(Tsf)
SF3;SF3(Tsf)
KEEPER; RETURN SCLR
EXPOSE:
KEEPER; IF Light OSHUT
KEEPER; CALL Integrate(ExpTime)
KEEPER; IF Light CSHUT
KEEPER; CALL Integrate(WaiTime)
KEEPER; RETURN EXPOSE
Integrate:
KEEPER; CALL IntFlush(275)
KEEPER; RETURN Integrate
IntFlush:
SR1;SR1(Tsrf1)
SR2;SR2(Tsrf2)
SR3;SR3(Tsrf3)
SR4;SR4(Tsrf4)
SR5;SR5(Tsrf5)
KEEPER
KEEPER;KEEPER(Tnpcf)
SR6;SR6(Tsrf6)
SR7;SR7(Tsrf7)
SR8;SR8(Tsrf8)
SR9;SR9(Tsrf9)
KEEPER; RETURN IntFlush
READOUT:
KEEPER; IF Npump CALL PUMP(Npump)
KE

Internally when we compile a timing script we get a list of lists, which represents instructions that happen at the same time. The CALL/GOTO/RETURN location targets are de-referenced and also the constant values

In [5]:
dsl_expr = compiler.construct_parser()
a = dsl_expr.parseString(compiler.tscript)

instructions = compiler.compile_timing_script()
print(instructions[:5])

[[[<ArchonInstructions.STATE: 0>, 'LowCLK'], [<ArchonInstructions.CONDITIONAL: 5>, False, 'LCLK'], [<ArchonInstructions.GOTO: 1>, 0]], [[<ArchonInstructions.STATE: 0>, 'IDLE'], [<ArchonInstructions.CALL: 2>, 8, 4]], [[<ArchonInstructions.STATE: 0>, 'KEEPER'], [<ArchonInstructions.CALL: 2>, 27, 'Expo']], [[<ArchonInstructions.STATE: 0>, 'KEEPER'], [<ArchonInstructions.CALL: 2>, 46, 'Expo']], [[<ArchonInstructions.STATE: 0>, 'KEEPER'], [<ArchonInstructions.CONDITIONAL: 5>, False, 'Nexpo'], [<ArchonInstructions.CALL: 2>, 6, 1]]]


# extracting state information
The `ArchonVM` class works using generator objects, so you can easily intervene and perform actions at any point in the timing you desire


In [6]:
#this gives us a generator expression
gen = vm.run()
#step through it like any other generator to get a sequnce of (time,instruction,statename) output by the timing cores
#each step is one archon core time step
for i in range(2):
    print(next(gen))

(0, 0, 'LowCLK')
(1, 0, 'LowCLK')


In [7]:
#nothing's happening!! Let's look at the debug output 
#need to use importlib.reload if in jupyter notebook or console
importlib.reload(logging)
vm.logger.addHandler(logging.StreamHandler())
vm.logger.setLevel(logging.DEBUG)
next(gen)
vm.logger.setLevel(logging.WARNING)

iptr 0: setting state LowCLK
DEBUG:archon_emu.ArchonVM:iptr 0: setting state LowCLK
iptr 0: CONDITIONAL on LCLK
DEBUG:archon_emu.ArchonVM:iptr 0: CONDITIONAL on LCLK
iptr 0: GOTO: 0
DEBUG:archon_emu.ArchonVM:iptr 0: GOTO: 0


In [10]:
#what's the current value of LCLK
print('value of LCLK is %d' % vm.parameters['LCLK'])
#lets set it to 1
vm.parameters['LCLK'] = 1
for i in range(6):
    print(next(gen))
#ok now it's working!

value of LCLK is 1
(9, 118, 'PFA')
(10, 118, 'PFA')
(11, 118, 'PFA')
(12, 118, 'PFA')
(13, 118, 'PFA')
(14, 118, 'PFA')


In [14]:
#can also query various internal state of the VM
print('iptr %d ' % vm.instruction_ptr)
print('loopcount %d' % vm.loopcount)
print('callstack %s' % str(vm.callstack))

#exhaust the current loop
for i in range(242):
    next(gen)
    
print('iptr %d ' % vm.instruction_ptr)
print('loopcount %d' % vm.loopcount)
print('callstack %s' % str(vm.callstack))



iptr 118 
loopcount 0
callstack [1, 8]
iptr 119 
loopcount 159
callstack [1, 8, 14]


# extracting clock values
